In [2]:
import os
from typing import Any, Dict, List, Tuple

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from pydantic import Field,BaseModel


In [ ]:
from reference_test import CASOS_TEST_PISA, PROMPT_JUEZ_CASO_ESTUDIO
from agente_pisa import agente_pisa #TODO chekear donde lo crea de de donde lo carga

In [5]:

# Cargamos las credenciales
load_dotenv(find_dotenv(usecwd=True))

# Inicialización del modelo para el agente juez
llm_juez = ChatGroq( temperature=0.0,model_name='openai/gpt-oss-120b')

Generamos un esquema estrucurado con pydantic de las respuestas que el juez debe mostrar a la salida de tal forma que se evita generar texto plano desestructurado

In [ ]:
#Definimos la estructura del output que el juez debe seguir
class VeredictoJuez(BaseModel):
    sin_alucinaciones: bool = Field(
        description='True si los datos numéricos y conclusiones coinciden con la referencia real. False si inventa datos o incumple guardrails.'
    )
    puntuacion_calidad: int = Field(
        ge=1, le=5, # greather than 1 less than 5
        description='Puntuación del 1 al 5. Nivel de alineación y rigor con los datos y conclusiones reales (1=totalmente contradictorio, 5=perfectamente alineado).'
    )
    completitud: int = Field(
        description='Puntuación del 1 al 5. ¿El agente respondió a todo lo solicitado en la pregunta de referencia?',
        ge=1, le=5,
    )
    motivo: str = Field(
        description='Justificación breve en una frase de por qué se considera acierto o fallo.'
    )

Pasamos la estructura creada al llm utilizado como Juez

In [5]:
llm_juez_estructurado = llm_juez.with_structured_output(VeredictoJuez)

## Función de evaluacion

unción de evaluación que conecta el agente, las herramientas y el juez

In [ ]:
def evaluar_caso_estudio(caso: dict) -> dict:

    #Cogemos las variables del caso
    test_id = caso["id"]
    pregunta = caso["pregunta"]
    herramientas_esperadas = caso.get("herramientas_imprescindibles", [])
    ref = caso["referencia"]
    
    print(f" Ejecutando caso: {test_id}...")
    
    # Ejecutar el agente PISA en un hilo aislado (util para evaluar diferentes casos)
    config_run = {"configurable": {"thread_id": f"eval_{test_id}"}, "recursion_limit": 10} #Configuración del hilo por caso
    salida = agente_pisa.invoke({"messages": [("user", pregunta)]}, config=config_run)
    mensajes = salida["messages"] #Obtenemos todos los mensajes del agente
    respuesta_agente = mensajes[-1].content #Nos quedamos con la respuesta 
    
    # Obtenemos las herramientas invocadas por agente_pisa
    herramientas_usadas = []
    for msg in mensajes:
        if hasattr(msg, "tool_calls") and msg.tool_calls: #Filtramos los obetos pydantic que tienen herramientas como atributo y con el campo de herramientas relleno
            for tc in msg.tool_calls:
                herramientas_usadas.append(tc["name"]) #Fuardamos las herramientas usadas

    ##ALGORITHM EVALUATOR            
    # Comprobamos si las herramientas llamadas por el agente son las esperadas
    if not herramientas_esperadas:
        tools_ok = len(herramientas_usadas) == 0
    else:
        # Deben haberse llamado todas las herramientas imprescindibles
        tools_ok = all(h in herramientas_usadas for h in herramientas_esperadas)
        
    # Formateamos los datos clave y las conclusiones de referencia en formato lista con - 
    datos_clave_str = "\n".join([f"- {d}" for d in ref.get("datos_clave", [])])
    conclusiones_str = "\n".join([f"- {c}" for c in ref.get("conclusiones_esperadas", [])])

    #
    # Rellenamos el prompt del juez
    prompt_completo = PROMPT_JUEZ_CASO_ESTUDIO.format(
        pregunta=pregunta,
        datos_clave=datos_clave_str,
        conclusiones_esperadas=conclusiones_str,
        respuesta_agente=respuesta_agente
    )
    
    veredicto: VeredictoJuez = llm_juez_estructurado.invoke([HumanMessage(content=prompt_completo)])
    
    # Devolvemos los resultados en un diccionario
    return {
        "ID": test_id,
        "Tools OK": "✅" if tools_ok else "❌",
        "Fidelidad OK": "✅" if veredicto.es_fiel_a_referencia else "❌",
        "Calidad (1-5)": veredicto.puntuacion_calidad,
        "Completitud (1-5)": veredicto.completitud,
        "Herramientas Esperadas": ", ".join(herramientas_esperadas),
        "Herramientas Usadas": ", ".join(herramientas_usadas) if herramientas_usadas else "Ninguna",
        "Dictamen Juez": veredicto.motivo
    }